In [277]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.window import Window

In [278]:
spark = SparkSession.builder \
	.appName("example") \
	.getOrCreate()	

In [279]:
df = spark.read.csv("retail_store_sales.csv",header=True)

In [280]:
#1.1. Загрузка и вывод схемы
df.printSchema()

root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Price Per Unit: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Total Spent: string (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Transaction Date: string (nullable = true)
 |-- Discount Applied: string (nullable = true)



In [281]:
#1.2. Очистка названий столбцов

In [282]:
df.columns

['Transaction ID',
 'Customer ID',
 'Category',
 'Item',
 'Price Per Unit',
 'Quantity',
 'Total Spent',
 'Payment Method',
 'Location',
 'Transaction Date',
 'Discount Applied']

In [283]:
for name in df.columns:
    df = df.withColumnRenamed(name, str(name.lower()).replace(" ","_"))

In [284]:
df.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|            category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|   TXN_9458126|    CUST_06|           Beverages| Item_16_

In [285]:
#1.3. Преобразование типов данных
for column in ["price_per_unit","quantity","total_spent"]:
    df = df.withColumn(column, F.when(df[column] != "NULL", df[column]).otherwise("0.0"))
    df = df.withColumn(column, col(column).cast("double"))

In [286]:
df.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|            category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|   TXN_9458126|    CUST_06|           Beverages| Item_16_

In [287]:
#2.1 Заполнение отсутствующие
df = df.withColumn("price_per_unit", F.when(col("price_per_unit") == "0.0",col("total_spent")/col("quantity")).otherwise(df["price_per_unit"]))

In [288]:
#2.3. Заполнение отсутствующих Quantity иTotal Spent
df = df.withColumn("quantity", F.when(col("quantity") == "0.0",col("total_spent")/col("price_per_unit")).otherwise(df["quantity"]))

In [289]:
df = df.withColumn("total_spent", F.when(col("total_spent") == "0.0",col("price_per_unit")*col("quantity")).otherwise(df["total_spent"]))

In [290]:
df.show()

+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|transaction_id|customer_id|            category|        item|price_per_unit|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------+-----------+--------------------+------------+--------------+--------+-----------+--------------+--------+----------------+----------------+
|   TXN_6867343|    CUST_09|          Patisserie| Item_10_PAT|          18.5|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|   TXN_3731986|    CUST_22|       Milk Products|Item_17_MILK|          29.0|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|   TXN_9303719|    CUST_02|            Butchers| Item_12_BUT|          21.5|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|   TXN_9458126|    CUST_06|           Beverages| Item_16_

In [291]:
#2.2. Восстановление отсутствующих Item
df_items = df["category","item","price_per_unit"].filter(df.item != "NULL").distinct()

In [292]:
df_items.show()

+--------------------+------------+--------------+
|            category|        item|price_per_unit|
+--------------------+------------+--------------+
|           Beverages| Item_16_BEV|          27.5|
|           Beverages|  Item_7_BEV|          14.0|
|Electric househol...| Item_23_EHE|          38.0|
|       Milk Products|Item_16_MILK|          27.5|
|Computers and ele...|  Item_1_CEA|           5.0|
|           Beverages| Item_25_BEV|          41.0|
|                Food|Item_12_FOOD|          21.5|
|           Furniture|  Item_4_FUR|           9.5|
|           Beverages| Item_17_BEV|          29.0|
|Computers and ele...|  Item_9_CEA|          17.0|
|           Beverages| Item_18_BEV|          30.5|
|                Food|Item_11_FOOD|          20.0|
|Computers and ele...|  Item_6_CEA|          12.5|
|Computers and ele...|  Item_8_CEA|          15.5|
|Electric househol...|  Item_6_EHE|          12.5|
|Electric househol...| Item_15_EHE|          26.0|
|           Beverages| Item_11_

In [293]:
df_items.sort(df.category, df.item).show()

+---------+-----------+--------------+
| category|       item|price_per_unit|
+---------+-----------+--------------+
|Beverages|Item_10_BEV|          18.5|
|Beverages|Item_11_BEV|          20.0|
|Beverages|Item_12_BEV|          21.5|
|Beverages|Item_13_BEV|          23.0|
|Beverages|Item_14_BEV|          24.5|
|Beverages|Item_15_BEV|          26.0|
|Beverages|Item_16_BEV|          27.5|
|Beverages|Item_17_BEV|          29.0|
|Beverages|Item_18_BEV|          30.5|
|Beverages|Item_19_BEV|          32.0|
|Beverages| Item_1_BEV|           5.0|
|Beverages|Item_20_BEV|          33.5|
|Beverages|Item_21_BEV|          35.0|
|Beverages|Item_22_BEV|          36.5|
|Beverages|Item_23_BEV|          38.0|
|Beverages|Item_24_BEV|          39.5|
|Beverages|Item_25_BEV|          41.0|
|Beverages| Item_2_BEV|           6.5|
|Beverages| Item_3_BEV|           8.0|
|Beverages| Item_4_BEV|           9.5|
+---------+-----------+--------------+
only showing top 20 rows



In [294]:
df_items = df_items.withColumnRenamed("item", "updated_item")

In [295]:
df_items.show()

+--------------------+------------+--------------+
|            category|updated_item|price_per_unit|
+--------------------+------------+--------------+
|           Beverages| Item_16_BEV|          27.5|
|           Beverages|  Item_7_BEV|          14.0|
|Electric househol...| Item_23_EHE|          38.0|
|       Milk Products|Item_16_MILK|          27.5|
|Computers and ele...|  Item_1_CEA|           5.0|
|           Beverages| Item_25_BEV|          41.0|
|                Food|Item_12_FOOD|          21.5|
|           Furniture|  Item_4_FUR|           9.5|
|           Beverages| Item_17_BEV|          29.0|
|Computers and ele...|  Item_9_CEA|          17.0|
|           Beverages| Item_18_BEV|          30.5|
|                Food|Item_11_FOOD|          20.0|
|Computers and ele...|  Item_6_CEA|          12.5|
|Computers and ele...|  Item_8_CEA|          15.5|
|Electric househol...|  Item_6_EHE|          12.5|
|Electric househol...| Item_15_EHE|          26.0|
|           Beverages| Item_11_

In [296]:
df1 = df.join(df_items, on=["category", "price_per_unit"], how="left").select("*")

In [297]:
df1.columns

['category',
 'price_per_unit',
 'transaction_id',
 'customer_id',
 'item',
 'quantity',
 'total_spent',
 'payment_method',
 'location',
 'transaction_date',
 'discount_applied',
 'updated_item']

In [298]:
df1.show()

+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+------------+
|            category|price_per_unit|transaction_id|customer_id|        item|quantity|total_spent|payment_method|location|transaction_date|discount_applied|updated_item|
+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09| Item_10_PAT|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True| Item_10_PAT|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|Item_17_MILK|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|Item_17_MILK|
|            Butchers|          21.5|   TXN_9303719|    CUST_02| Item_12_BUT|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           

In [299]:
df1 = df1.withColumn("item",col("updated_item"))

In [301]:
df1 = df1.drop("updated_item")

In [302]:
df1.show()

+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|            category|price_per_unit|transaction_id|customer_id|        item|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09| Item_10_PAT|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|Item_17_MILK|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|            Butchers|          21.5|   TXN_9303719|    CUST_02| Item_12_BUT|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|           Beverages|          27.5|   TXN_9458126|    CU

In [172]:
#2.4. Удалите оставшийся строки с пропусками

In [303]:
df1 = df1.filter((df1.quantity!=0.0) & (df1.price_per_unit != 0.0) & (df1.total_spent != 0.0) & (df1.category != "NULL"))

In [306]:
df1.show()

+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|            category|price_per_unit|transaction_id|customer_id|        item|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09| Item_10_PAT|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|Item_17_MILK|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|            Butchers|          21.5|   TXN_9303719|    CUST_02| Item_12_BUT|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|           Beverages|          27.5|   TXN_9458126|    CU

In [307]:
#3.1. Самые популярные категории товаров
df_sorted = df1.groupBy("category").agg(F.sum("quantity").alias("sales")).orderBy("sales", ascending=False)

In [310]:
df_sorted.show(5)

+--------------------+------+
|            category| sales|
+--------------------+------+
|           Furniture|8462.0|
|                Food|8387.0|
|           Beverages|8358.0|
|       Milk Products|8339.0|
|Electric househol...|8309.0|
+--------------------+------+
only showing top 5 rows



In [315]:
#3.2. Анализ среднего чека

In [311]:
df_payment_method = df1.groupBy("payment_method").agg(F.avg("total_spent").alias("total_spent")).orderBy("total_spent", ascending=False)

In [312]:
df_payment_method = df_payment_method.withColumn("total_spent",F.round(df_payment_method.total_spent,2))

In [313]:
df_payment_method.show()

+--------------+-----------+
|payment_method|total_spent|
+--------------+-----------+
|          Cash|     131.05|
|   Credit Card|     129.13|
|Digital Wallet|     128.72|
+--------------+-----------+



In [226]:
df_location = df1.groupBy("location").agg(F.avg("total_spent").alias("total_spent")).orderBy("total_spent", ascending=False)

In [227]:
df_location = df_location.withColumn("total_spent",F.round(df_location.total_spent,2))

In [228]:
df_location.show()

+--------+-----------+
|location|total_spent|
+--------+-----------+
|  Online|     130.42|
|In-store|     128.86|
+--------+-----------+



In [316]:
df1.show()

+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|            category|price_per_unit|transaction_id|customer_id|        item|quantity|total_spent|payment_method|location|transaction_date|discount_applied|
+--------------------+--------------+--------------+-----------+------------+--------+-----------+--------------+--------+----------------+----------------+
|          Patisserie|          18.5|   TXN_6867343|    CUST_09| Item_10_PAT|    10.0|      185.0|Digital Wallet|  Online|      2024-04-08|            True|
|       Milk Products|          29.0|   TXN_3731986|    CUST_22|Item_17_MILK|     9.0|      261.0|Digital Wallet|  Online|      2023-07-23|            True|
|            Butchers|          21.5|   TXN_9303719|    CUST_02| Item_12_BUT|     2.0|       43.0|   Credit Card|  Online|      2022-10-05|           False|
|           Beverages|          27.5|   TXN_9458126|    CU

In [318]:
#4.1. Временные признаки

In [244]:
df1 = df1.withColumn("day_of_week",F.dayofweek(col("transaction_date"))).withColumn("transaction_month",F.month(col("transaction_date")))

In [248]:
#4.2. Продажи по дням недели
df1.groupBy("day_of_week").agg(F.avg("total_spent").alias("total_spent")).orderBy("day_of_week").show()

+-----------+------------------+
|day_of_week|       total_spent|
+-----------+------------------+
|          1|124.78796245168415|
|          2| 119.8484251968504|
|          3|122.84675615212528|
|          4|119.68119139547711|
|          5|122.55893854748604|
|          6| 128.7533185840708|
|          7|125.45663122551763|
+-----------+------------------+



In [251]:
#4.3.Продажи по месяцам
df1.groupBy("transaction_month").agg(F.avg("total_spent").alias("total_spent")).orderBy("transaction_month").show()

+-----------------+------------------+
|transaction_month|       total_spent|
+-----------------+------------------+
|                1| 128.1565025716385|
|                2|124.02590673575129|
|                3| 120.1099116781158|
|                4|126.12299196787149|
|                5|120.61423039690223|
|                6|125.02023121387283|
|                7|120.76124885215795|
|                8|118.65976900866218|
|                9|126.68364348677767|
|               10|122.09969325153374|
|               11|121.98055832502493|
|               12|125.54549854791868|
+-----------------+------------------+



In [252]:
#4.4. Признаки клиента

In [256]:
df_customer_lifetime_value = df1.groupBy("customer_id").agg(F.sum("total_spent").alias("customer_lifetime_value")).orderBy("customer_lifetime_value", ascending=False)

In [258]:
df_customer_lifetime_value.show()

+-----------+-----------------------+
|customer_id|customer_lifetime_value|
+-----------+-----------------------+
|    CUST_24|                68452.0|
|    CUST_08|                67351.5|
|    CUST_05|                66974.5|
|    CUST_16|                65570.5|
|    CUST_13|                65037.0|
|    CUST_23|                64507.0|
|    CUST_10|                63155.5|
|    CUST_15|                63117.5|
|    CUST_21|                62933.0|
|    CUST_02|                62046.5|
|    CUST_04|                61767.5|
|    CUST_22|                61732.5|
|    CUST_20|                61533.0|
|    CUST_09|                61423.5|
|    CUST_12|                61360.0|
|    CUST_03|                60811.0|
|    CUST_19|                60797.0|
|    CUST_11|                60733.0|
|    CUST_07|                60694.5|
|    CUST_14|                60528.0|
+-----------+-----------------------+
only showing top 20 rows



In [271]:
window = Window.orderBy(col("customer_lifetime_value").desc())

In [275]:
df_customer_lifetime_value_ranked = df_customer_lifetime_value.withColumn("drn", F.dense_rank().over(window))

In [276]:
df_customer_lifetime_value_ranked.filter(col("drn") <= 10).show()

+-----------+-----------------------+---+
|customer_id|customer_lifetime_value|drn|
+-----------+-----------------------+---+
|    CUST_24|                68452.0|  1|
|    CUST_08|                67351.5|  2|
|    CUST_05|                66974.5|  3|
|    CUST_16|                65570.5|  4|
|    CUST_13|                65037.0|  5|
|    CUST_23|                64507.0|  6|
|    CUST_10|                63155.5|  7|
|    CUST_15|                63117.5|  8|
|    CUST_21|                62933.0|  9|
|    CUST_02|                62046.5| 10|
+-----------+-----------------------+---+

